# 1. 1. Exception 문법 및 기본 구조

예외 처리는 주로 `try`, `except`, `else`, `finally` 블록으로 구성됩니다. 예외 발생 여부에 따라 어떤 블록이 실행되는지 정확히 이해하는 것이 중요합니다.

> [!tip] 블록별 목적 및 권장 사항
> 
> - **`try`**: 예외 발생 가능성이 있는 코드만 최소한으로 배치합니다.
>     
> - **`except`**: 구체적인 예외(예: `ValueError`)를 먼저 잡고, 포괄적인 예외(`Exception`)는 가장 마지막에 배치해야 합니다.
>     
> - **`else`**: `try` 블록이 성공했을 때만 실행됩니다. `try` 블록 비대화를 방지합니다.
>     
> - **`finally`**: 에러 발생 여부와 무관하게 `리소스 정리(DB 연결 종료, 파일 닫기 등)`를 보장합니다.
>     

In [6]:
def safe_divide(a, b):
    """나눗셈 연산을 안전하게 수행하며 예외 처리 흐름을 보여주는 함수"""
    print(f"\n--- [실행] {a} / {b} 연산 시작 ---")
    try:
        ## 예외가 발생할 가능성이 있는 코드
        result = a / b
    except ZeroDivisionError as e:
        ## 특정 예외가 발생했을 때 실행되는 코드
        print(f"[Except 블록] 에러 발생! 0으로 나눌 수 없습니다.\n(원인: {e})")
        result = None
    except (TypeError, ValueError) as e:
        ## 여러 예외를 튜플로 묶어서 처리 가능
        print(f"[Except 블록] 에러 발생! 잘못된 타입이나 값이 입력되었습니다.\n(원인: {e})")
        result = None
    except Exception as e:
        ## 모든 예외의 최상위 클래스 (BaseException 제외)
        print(f"[Except 블록] 알 수 없는 오류 발생: {e}")
        result = None
    else:
        ## try 블록에서 예외가 발생하지 않았을 때만! 실행되는 코드
        print(f"[Else 블록] 예외 없이 정상 처리되었습니다. 결과: {result}")
    finally:
        ## 무조건! 실행되는 코드 (리소스 정리)
        print("[Finally 블록] 연산 종료. 리소스를 정리합니다.")
        
    return result

## 학습용 실행 예시
safe_divide(10, 2)    ## 정상 케이스: try -> else -> finally 실행
safe_divide(10, 0)    ## 에러 케이스: try -> except(ZeroDivisionError) -> finally 실행
safe_divide(10, "A")  ## 에러 케이스: try -> except(TypeError) -> finally 실행


--- [실행] 10 / 2 연산 시작 ---
[Else 블록] 예외 없이 정상 처리되었습니다. 결과: 5.0
[Finally 블록] 연산 종료. 리소스를 정리합니다.

--- [실행] 10 / 0 연산 시작 ---
[Except 블록] 에러 발생! 0으로 나눌 수 없습니다.
(원인: division by zero)
[Finally 블록] 연산 종료. 리소스를 정리합니다.

--- [실행] 10 / A 연산 시작 ---
[Except 블록] 에러 발생! 잘못된 타입이나 값이 입력되었습니다.
(원인: unsupported operand type(s) for /: 'int' and 'str')
[Finally 블록] 연산 종료. 리소스를 정리합니다.


# 2. Exception 객체의 구성 요소 및 속성

예외가 발생하면 Python은 예외 클래스의 인스턴스(객체)를 생성합니다.

이 객체는 에러 추적 및 디버깅에 필요한 중요한 내부 속성(Attributes)을 가집니다.

> [!info] 주요 Dunder(Double Underscore) 속성
> 
> - `__traceback__`: 콜스택 정보를 담은 Traceback 객체
>     
> - `__context__`: 암시적 예외 체이닝 (에러 처리 중 발생한 또 다른 에러의 원본)
>     
> - `__cause__`: 명시적 예외 체이닝 (`raise ... from ...` 사용 시 원본 에러)
>     

In [9]:
def inspect_exception_attributes():
    """예외 객체의 내부 속성(__context__, __cause__, args 등)을 확인하는 예제"""
    try:
        try:
            ## 1. 최초 에러 발생
            1 / 0
        except ZeroDivisionError as inner_e:
            ## 2. 에러 처리 중 또 다른 에러 발생 (암시적 체이닝)
            raise ValueError("내부 연산 실패로 인한 값 오류 발생")
    
    except ValueError as outer_e:
        print("\n--- Exception 속성 분석 ---")
        
        ## args: 예외 메시지 튜플
        print(f"1. e.args: {outer_e.args}")

        ## __context__: except 블록 내부에서 다른 예외가 발생했을 때, 원래 예외를 담음
        print(f"2. e.__context__: {type(outer_e.__context__).__name__} 객체가 담겨있음")

        ## __traceback__: 콜스택 정보
        print(f"3. e.__traceback__: {outer_e.__traceback__}")

        ## Python 3.11+ 추가 기능: add_note
        if hasattr(outer_e, 'add_note'):
            outer_e.add_note("--> 디버깅 노트: 이 에러는 학습용으로 의도적으로 발생시켰습니다.")
            print("4. e.add_note()를 사용하여 커스텀 디버깅 노트를 추가했습니다.")

inspect_exception_attributes()
"""
--- Exception 속성 분석 ---
1. e.args: ('내부 연산 실패로 인한 값 오류 발생',)
2. e.__context__: ZeroDivisionError 객체가 담겨있음
3. e.__traceback__: <traceback object at 0x0000024F099C2040>
4. e.add_note()를 사용하여 커스텀 디버깅 노트를 추가했습니다.
"""


--- Exception 속성 분석 ---
1. e.args: ('내부 연산 실패로 인한 값 오류 발생',)
2. e.__context__: ZeroDivisionError 객체가 담겨있음
3. e.__traceback__: <traceback object at 0x0000024F099C2040>
4. e.add_note()를 사용하여 커스텀 디버깅 노트를 추가했습니다.


# 3. Error 종류 및 자주 발생하는 에러 실습


| **예외 이름 (Exception Name)** | **상위 클래스**      | **발생 원인 / 설명**                          |
| -------------------------- | --------------- | --------------------------------------- |
| **`BaseException`**        | -               | 모든 내장 예외의 최상위 클래스                       |
| **`SystemExit`**           | `BaseException` | `sys.exit()` 함수가 호출될 때 프로그램 종료를 위해 발생   |
| **`KeyboardInterrupt`**    | `BaseException` | 사용자가 실행을 중단할 때 발생 (예: `Ctrl + C` 입력)    |
| **`GeneratorExit`**        | `BaseException` | 제너레이터나 코루틴이 닫힐 때 (`close()` 호출 시) 발생    |
| **`Exception`**            | `BaseException` | 시스템 종료 관련을 제외한 **모든 일반 내장 예외의 최상위 클래스** |

#### 3.2 자주 발생하는 일반 예외

코드 작성 및 실행 과정에서 가장 빈번하게 마주치는 핵심 예외들입니다.

|**예외 이름 (Exception Name)**|**상위 클래스**|**발생 원인 / 설명**|
|---|---|---|
|**`AttributeError`**|`Exception`|객체에 없는 속성(Attribute)이나 메서드를 참조·대입하려 할 때 발생|
|**`NameError`**|`Exception`|로컬 또는 글로벌 변수 이름을 찾을 수 없을 때 (선언되지 않은 변수 사용 시) 발생|
|**`TypeError`**|`Exception`|연산이나 함수가 부적절한 데이터 타입 객체에 적용될 때 (예: `str` + `int`) 발생|
|**`ValueError`**|`Exception`|데이터 타입은 올바르나, 값이 적절하지 않을 때 (예: `int('abc')`) 발생|

#### 3.3 시퀀스 및 매핑 예외 (자료구조 관련)

리스트, 튜플, 딕셔너리 등 자료구조를 다룰 때 발생하는 예외입니다.

|**예외 이름 (Exception Name)**|**상위 클래스**|**발생 원인 / 설명**|
|---|---|---|
|**`LookupError`**|`Exception`|매핑 또는 시퀀스에서 키나 인덱스가 잘못되었을 때의 상위 클래스|
|**`IndexError`**|`LookupError`|시퀀스(리스트, 튜플 등)의 범위를 벗어난 인덱스를 참조할 때 발생|
|**`KeyError`**|`LookupError`|딕셔너리에서 존재하지 않는 키(Key)를 참조할 때 발생|

#### 3.4 산술 및 연산 예외

수학적 연산 과정에서 규칙을 위배했을 때 발생합니다.

|**예외 이름 (Exception Name)**|**상위 클래스**|**발생 원인 / 설명**|
|---|---|---|
|**`ArithmeticError`**|`Exception`|다양한 산술 오류에 대한 상위 클래스|
|**`ZeroDivisionError`**|`ArithmeticError`|숫자를 0으로 나누려 할 때 발생 (`/` 또는 `%` 연산)|
|**`OverflowError`**|`ArithmeticError`|산술 연산 결과가 표현할 수 있는 최대 범위를 초과할 때 발생|

#### 3.5 파일 및 입력/출력(I/O) 예외

외부 시스템, 파일 시스템과의 상호작용 과정에서 발생하는 예외입니다.

|**예외 이름 (Exception Name)**|**상위 클래스**|**발생 원인 / 설명**|
|---|---|---|
|**`OSError`**|`Exception`|시스템 관련 오류의 상위 클래스 (구 버전의 `IOError` 통합)|
|**`FileNotFoundError`**|`OSError`|정의된 경로에 파일이나 디렉토리가 존재하지 않을 때 발생|
|**`PermissionError`**|`OSError`|파일 시스템 접근 권한이 없을 때 발생 (예: 쓰기 금지 파일에 쓰기 시도)|
|**`FileExistsError`**|`OSError`|이미 존재하는 파일이나 디렉토리를 새로 만들려고 할 때 발생|

#### 3.6 구문 및 모듈 로딩 예외

코드의 구조적 문제나 외부 라이브러리를 불러올 때 발생합니다.

| **예외 이름 (Exception Name)** | **상위 클래스**    | **발생 원인 / 설명**                                  |
| -------------------------- | ------------- | ----------------------------------------------- |
| **`SyntaxError`**          | `Exception`   | 파이썬 문법 규칙에 맞지 않는 코드가 있을 때 파싱 단계에서 발생            |
| **`IndentationError`**     | `SyntaxError` | 들여쓰기(Indent)가 잘못되었을 때 발생                        |
| **`ImportError`**          | `Exception`   | `import` 문이 모듈을 로드하는 데 실패했거나 정의된 이름을 찾지 못할 때 발생 |
| 

In [10]:
def common_errors_practice():
    print("\n--- 자주 발생하는 에러 실습 ---")

    ## 1. TypeError: 타입이 맞지 않는 연산
    try:
        result = "문자열" + 5
    except TypeError as e:
        print(f"TypeError 포착: {e}")

    ## 2. ValueError: 타입은 맞지만 값이 유효하지 않음
    try:
        result = int("숫자아님")
    except ValueError as e:
        print(f"ValueError 포착: {e}")

    ## 3. KeyError: 딕셔너리에 존재하지 않는 키 접근
    my_dict = {"name": "Alice"}
    try:
        result = my_dict["age"]
    except KeyError as e:
        print(f"KeyError 포착: {e} 키가 존재하지 않습니다.")

    ## 4. IndexError: 리스트 등의 인덱스 범위 초과
    my_list = [1, 2, 3]
    try:
        result = my_list[10]
    except IndexError as e:
        print(f"IndexError 포착: {e}")

    ## 5. AttributeError: 객체에 없는 메서드나 속성 접근   
    try:
        result = my_list.append_item(4) ## append()가 올바른 메서드임
    except AttributeError as e:
        print(f"AttributeError 포착: {e}")

common_errors_practice()
"""
--- 자주 발생하는 에러 실습 ---
TypeError 포착: can only concatenate str (not "int") to str
ValueError 포착: invalid literal for int() with base 10: '숫자아님'
KeyError 포착: 'age' 키가 존재하지 않습니다.
IndexError 포착: list index out of range
AttributeError 포착: 'list' object has no attribute 'append_item'
"""


--- 자주 발생하는 에러 실습 ---
TypeError 포착: can only concatenate str (not "int") to str
ValueError 포착: invalid literal for int() with base 10: '숫자아님'
KeyError 포착: 'age' 키가 존재하지 않습니다.
IndexError 포착: list index out of range
AttributeError 포착: 'list' object has no attribute 'append_item'


# 4.1 예외 체이닝 (Exception Chaining)과 `__suppress_context__`

어떤 예외를 처리하는 과정에서 의미를 명확히 하기 위해 새로운 예외로 변환하여 던질 때 사용합니다. `from` 키워드를 사용하면 명시적 체이닝이 발생합니다.

> [!quote] 체이닝 패턴
> 
> - **보존 (`from e`)**: 로그 트래킹을 위해 원본 에러 정보를 유지해야 할 때.
>     
> - **숨김 (`from None`)**: 보안상 원본 에러(예: DB 연결 문자열 에러)를 감추고 싶을 때 (`__suppress_context__`가 `True`로 설정됨).
>     

In [14]:
def process_database_data():
    try:
        raise ConnectionError("DB 서버 응답 없음")
    except ConnectionError as e:
        ## 1. 원래 에러(e)를 원인으로 명시하여 새로운 에러 발생 (권장)
        raise RuntimeError("데이터 처리 중 심각한 오류가 발생했습니다.") from e

def process_and_hide_error():
    try:
        raise ValueError("숨기고 싶은 내부 비밀번호 검증 에러")
    except ValueError:
        ## 2. 이전 에러의 흔적을 완전히 지우고 싶을 때
        raise RuntimeError("단순 인증 실패입니다.") from None

# 4.2 사용자 정의 예외 (Custom Exceptions)

도메인 주도 설계(DDD)나 비즈니스 로직의 명확성을 위해

`Exception`을 상속받아 커스텀 예외를 생성합니다.

상태(State) 값을 객체에 담아 상위 호출자에게 유용한 컨텍스트를 제공할 수 있습니다.

In [16]:
class InsufficientFundsError(Exception):
    """잔고 부족 시 발생하는 사용자 정의 예외"""

    def __init__(self, current_balance, request_amount):
        self.current_balance = current_balance
        self.request_amount = request_amount
        self.shortfall = request_amount - current_balance

        message = f"출금 실패: 잔고가 {self.shortfall}원 부족합니다. (현재: {current_balance}원)"
        super().__init__(message)

def withdraw(amount):
    balance = 1000
    if amount > balance:
        raise InsufficientFundsError(balance, amount)
    print("출금 성공")

try:
    withdraw(5000)
except InsufficientFundsError as e:
    ## 커스텀 예외의 속성(shortfall)을 활용하여 구체적인 로직 처리
    print(f"[경고] {e.shortfall}원 필요합니다.")

[경고] 4000원 필요합니다.


# 4.3 예외 그룹 (ExceptionGroup)

> [!note] Python 3.11+ 신규 기능
> 
> 비동기 프로그래밍(`asyncio`)이나 병렬 처리에서 **여러 예외가 동시에 발생**했을 때, 이를 하나의 객체로 묶어 던지고 `except*` 구문으로 타입별 필터링 처리를 가능하게 합니다.

In [18]:
def test_exception_group():
    try:
        errs = [ValueError("잘못된 값 1"), TypeError("타입 에러"), ValueError("잘못된 값 2")]

        raise ExceptionGroup("병렬 처리 중 다중 에러 발생", errs)

    except* ValueError as e:
        ## e.exceptions에는 ValueError 인스턴스 2개만 리스트 형태로 묶임
        print(f"[ExceptionGroup] ValueError 그룹 처리: {e.exceptions}")

    except* TypeError as e:
        ## e.exceptions에는 TypeError 인스턴스 1개만 묶임
        print(f"[ExceptionGroup] TypeError 그룹 처리: {e.exceptions}")

test_exception_group()

[ExceptionGroup] ValueError 그룹 처리: (ValueError('잘못된 값 1'), ValueError('잘못된 값 2'))
[ExceptionGroup] TypeError 그룹 처리: (TypeError('타입 에러'),)


# 4.4 traceback 모듈의 실무 활용

실제 운영 환경(Production)에서는 `print()` 대신 에러의 상세 내역(Traceback)을 파일이나 모니터링 시스템에 로깅해야 합니다.

이때 `traceback.format_exc()`가 핵심적인 역할을 합니다.

In [19]:
import traceback

def complex_operation():
    return 1 / 0

try:
    complex_operation()
except Exception as e:
    ## 프로그램 강제 종료 없이 Traceback 전체 텍스트를 추출
    error_detail = traceback.format_exc()

    ## 실제 환경에서는 logger.error() 등으로 처리
    print("--- [서버 로깅 시스템] 에러 로그 ---")
    print(f"요약 메시지: {e}")
    print(f"상세 스택 트레이스:\n{error_detail}")

--- [서버 로깅 시스템] 에러 로그 ---
요약 메시지: division by zero
상세 스택 트레이스:
Traceback (most recent call last):
  File "C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_88092\1036894402.py", line 7, in <module>
    complex_operation()
  File "C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_88092\1036894402.py", line 4, in complex_operation
    return 1 / 0
           ~~^~~
ZeroDivisionError: division by zero

